In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import json
from itertools import product
from collections import Counter
import os
from Bio.Seq import reverse_complement


In [3]:
output_path = '/home/nika/Documents/barpwml/genomes'
with open (output_path, 'r') as f:
    genomes = json.load(f)
print(f'reloaded {len(genomes)} genomes')

reloaded 95 genomes


In [4]:
def get_all_kmers(k):
    # Generate ALL possible k-mers for DNA alphabet.
    # This defines the fixed vocabulary — every genome gets score against the same list, even if some k-mers never appear
    bases = ['A', 'C', 'G', 'T']
    return [''.join(p) for p in product(bases, repeat=k)]

In [ ]:
def count_kmers(sequence, k=4, include_reverse_complement=True):
    """
    Count k-mer frequencies in a DNA sequence.

    Parameters:
        sequence: DNA string (uppercase)
        k: k-mer length
        include_reverse_complement: if True, also count the reverse strand

    Returns:
        dict mapping each possible k-mer to its relative frequency
    """

    # Remove N and any non-ATGC characters
    sequence = sequence.upper()
    sequence = ''.join(b for b in sequence if b in 'ATGC')

    if len(sequence) < k:
        raise ValueError(f'Sequence too short ({len(sequence)} bp for k={k}')

    # Build the complete vocabulary (all 4^k possible k-mers)
    vocabulary = get_all_kmers(k)
    counts = Counter({kmer: 0 for kmer in vocabulary})  # start all at 0

    # Count k-mers on the forward strand
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i : i+k]
        if kmer in counts:
            counts[kmer] += 1
    
    # Count reverse complement strand too
    if include_reverse_complement:
        rev_comp = reverse_complement(sequence)
        for i in range(len(rev_comp) - k + 1):
            kmer = rev_comp[i : i+k]
            if kmer in counts:
                counts[kmer] += 1
    
    # Convert to relative frequencies
    total = sum(counts.values())
    frequencies = {kmer: counts[kmer] / total for kmer in vocabulary}
    
    return frequencies


In [15]:
def reverse_complement(sequence):
    # Return the reverse complement of a DNA sequence
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement[b] for b in reversed(sequence))
